# Suite ADJ — Grid adjacency

An adjacency spec from one child to another moves the other beside it, along its rotation quantized to a quarter turn, by one change of the grid's sequence, or into the cell beside it that a spec along the same side placed, which they then share; removing the spec reverts the child to its original insertion position. Each step is one mutation, a transaction expression the bench journals, and the container layout's adjustments to it, in order, which the step asserts; the suite's last case replays the journal in a fresh store, and finds the same adjustments. mbse-basis [CAP-239: grid adjacency](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-239.md), by the rules of [the diagrams proposal](https://github.com/pitaman71/mbse-basis/blob/main/proposals/DIAGRAMS.md).

In [ ]:
from support import Bench, text
from mbse.Diagrams.Framework import Container, Logical

b = Bench()

<a id="ADJ-01"></a>
## ADJ-01 · A rotation is quantized to the nearest quarter turn, half way to the clockwise one

In [ ]:
assert text([Container.quarter(d) for d in (0.0, 44.9, 45.0, 134.9, 190.0, 270.0, 315.0, -90.0, 359.5, 720.0)]) == '["north","north","east","east","south","west","north","west","north","north"]'

<a id="ADJ-02"></a>
## ADJ-02 · A spec moves its child beside the other, by one change of the sequence, and the rule is met

In [ ]:
assert b.child('', 'cpu') == '[{"kind":"grid","container":"overview","span":3,"grows":"rows","cell_rotation":0.0},{"kind":"place","shape":"cpu","order":0,"to":[0,0]},{"kind":"extents","rows":1,"columns":1}]'
assert b.layout('cpu', 3) == '[{"kind":"grid","container":"cpu","span":3,"grows":"rows","cell_rotation":0.0}]'
assert b.child('cpu', 'a') == '[{"kind":"place","shape":"a","order":0,"to":[0,0]},{"kind":"extents","rows":1,"columns":1}]'
assert b.child('cpu', 'b') == '[{"kind":"place","shape":"b","order":1,"to":[0,1]},{"kind":"extents","rows":1,"columns":2}]'
assert b.child('cpu', 'c') == '[{"kind":"place","shape":"c","order":2,"to":[0,2]},{"kind":"extents","rows":1,"columns":3}]'
assert b.child('cpu', 'd') == '[{"kind":"place","shape":"d","order":3,"to":[1,0]},{"kind":"extents","rows":2,"columns":3}]'
assert b.child('cpu', 'e') == '[{"kind":"place","shape":"e","order":4,"to":[1,1]}]'
assert b.spec('cpu', 'a', 'e', 90.0) == '[{"kind":"move","shape":"e","from":[1,1],"to":[0,1]},{"kind":"move","shape":"b","from":[0,1],"to":[0,2]},{"kind":"move","shape":"c","from":[0,2],"to":[1,0]},{"kind":"move","shape":"d","from":[1,0],"to":[1,1]},{"kind":"rule","from_shape":"a","to_shape":"e","degrees":90.0,"status":"met"}]'
assert b.spec('cpu', 'a', 'd', 190.0) == '[{"kind":"move","shape":"d","from":[1,1],"to":[1,0]},{"kind":"move","shape":"c","from":[1,0],"to":[1,1]},{"kind":"rule","from_shape":"a","to_shape":"d","degrees":190.0,"status":"met"}]'
assert b.cells("cpu") == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["e",4]],null,1,1],[[0,2],[["b",1]],null,1,1],[[1,0],[["d",3]],null,1,1],[[1,1],[["c",2]],null,1,1]]'

<a id="ADJ-03"></a>
## ADJ-03 · Two specs from one child along one side share a cell, which grows along its grid's axis

In [ ]:
assert b.spec('cpu', 'a', 'c', 80.0) == '[{"kind":"move","shape":"c","from":[1,1],"to":[0,1]},{"kind":"rule","from_shape":"a","to_shape":"c","degrees":80.0,"status":"met"}]'
assert b.cells("cpu") == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["e",4],["c",2]],"rows",2,1],[[0,2],[["b",1]],null,1,1],[[1,0],[["d",3]],null,1,1]]'
assert b.child('', 'io') == '[{"kind":"place","shape":"io","order":1,"to":[0,1]},{"kind":"extents","rows":1,"columns":2}]'
assert b.layout('io', 3) == '[{"kind":"grid","container":"io","span":3,"grows":"rows","cell_rotation":0.0}]'
assert b.child('io', 'p') == '[{"kind":"place","shape":"p","order":0,"to":[0,0]},{"kind":"extents","rows":1,"columns":1}]'
assert b.child('io', 'q') == '[{"kind":"place","shape":"q","order":1,"to":[0,1]},{"kind":"extents","rows":1,"columns":2}]'
assert b.child('io', 'r') == '[{"kind":"place","shape":"r","order":2,"to":[0,2]},{"kind":"extents","rows":1,"columns":3}]'
assert b.child('io', 's') == '[{"kind":"place","shape":"s","order":3,"to":[1,0]},{"kind":"extents","rows":2,"columns":3}]'
assert b.child('io', 't') == '[{"kind":"place","shape":"t","order":4,"to":[1,1]}]'
assert b.spec('io', 'p', 'r', 180.0) == '[{"kind":"move","shape":"s","from":[1,0],"to":[0,2]},{"kind":"move","shape":"r","from":[0,2],"to":[1,0]},{"kind":"rule","from_shape":"p","to_shape":"r","degrees":180.0,"status":"met"}]'
assert b.spec('io', 'p', 't', 180.0) == '[{"kind":"move","shape":"t","from":[1,1],"to":[1,0]},{"kind":"rule","from_shape":"p","to_shape":"t","degrees":180.0,"status":"met"}]'
assert b.cells("io") == '[[[0,0],[["p",0]],null,1,1],[[0,1],[["q",1]],null,1,1],[[0,2],[["s",3]],null,1,1],[[1,0],[["r",2],["t",4]],"rows",2,1]]'

<a id="ADJ-04"></a>
## ADJ-04 · Removing a spec drops its rule and reverts its child to its original insertion position, keeping the specs to it that remain

In [ ]:
assert b.unspec('cpu', 'a', 'e') == '[{"kind":"rule","from_shape":"a","to_shape":"e","degrees":90.0,"status":"dropped"},{"kind":"move","shape":"e","from":[0,1],"to":[1,1]}]'
assert b.cells("cpu") == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["c",2]],null,1,1],[[0,2],[["b",1]],null,1,1],[[1,0],[["d",3]],null,1,1],[[1,1],[["e",4]],null,1,1]]'
assert b.unspec('cpu', 'a', 'c') == '[{"kind":"rule","from_shape":"a","to_shape":"c","degrees":80.0,"status":"dropped"},{"kind":"move","shape":"b","from":[0,2],"to":[0,1]},{"kind":"move","shape":"c","from":[0,1],"to":[0,2]}]'
assert b.cells("cpu") == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["b",1]],null,1,1],[[0,2],[["c",2]],null,1,1],[[1,0],[["d",3]],null,1,1],[[1,1],[["e",4]],null,1,1]]'
assert b.child('', 'pcb') == '[{"kind":"place","shape":"pcb","order":2,"to":[0,2]},{"kind":"extents","rows":1,"columns":3}]'
assert b.layout('pcb', 3) == '[{"kind":"grid","container":"pcb","span":3,"grows":"rows","cell_rotation":0.0}]'
assert b.child('pcb', 'u') == '[{"kind":"place","shape":"u","order":0,"to":[0,0]},{"kind":"extents","rows":1,"columns":1}]'
assert b.child('pcb', 'v') == '[{"kind":"place","shape":"v","order":1,"to":[0,1]},{"kind":"extents","rows":1,"columns":2}]'
assert b.child('pcb', 'w') == '[{"kind":"place","shape":"w","order":2,"to":[0,2]},{"kind":"extents","rows":1,"columns":3}]'
assert b.child('pcb', 'x') == '[{"kind":"place","shape":"x","order":3,"to":[1,0]},{"kind":"extents","rows":2,"columns":3}]'
assert b.spec('pcb', 'u', 'x', 90.0) == '[{"kind":"move","shape":"x","from":[1,0],"to":[0,1]},{"kind":"move","shape":"v","from":[0,1],"to":[0,2]},{"kind":"move","shape":"w","from":[0,2],"to":[1,0]},{"kind":"rule","from_shape":"u","to_shape":"x","degrees":90.0,"status":"met"}]'
assert b.spec('pcb', 'v', 'x', 180.0) == '[{"kind":"rule","from_shape":"v","to_shape":"x","degrees":180.0,"status":"ignored"}]'
assert b.unspec('pcb', 'u', 'x') == '[{"kind":"rule","from_shape":"u","to_shape":"x","degrees":90.0,"status":"dropped"},{"kind":"move","shape":"v","from":[0,2],"to":[0,1]},{"kind":"move","shape":"w","from":[1,0],"to":[0,2]},{"kind":"move","shape":"x","from":[0,1],"to":[1,0]}]'
assert b.cells("pcb") == '[[[0,0],[["u",0]],null,1,1],[[0,1],[["v",1]],null,1,1],[[0,2],[["w",2]],null,1,1],[[1,0],[["x",3]],null,1,1]]'
assert b.rules("pcb") == '[["v","x",180.0,true]]'

<a id="ADJ-05"></a>
## ADJ-05 · A spec the grid cannot meet is ignored, and kept

In [ ]:
assert b.child('', 'bus') == '[{"kind":"place","shape":"bus","order":3,"to":[1,0]},{"kind":"extents","rows":2,"columns":3}]'
assert b.layout('bus', 3) == '[{"kind":"grid","container":"bus","span":3,"grows":"rows","cell_rotation":0.0}]'
assert b.child('bus', 'a') == '[{"kind":"place","shape":"a","order":0,"to":[0,0]},{"kind":"extents","rows":1,"columns":1}]'
assert b.child('bus', 'b') == '[{"kind":"place","shape":"b","order":1,"to":[0,1]},{"kind":"extents","rows":1,"columns":2}]'
assert b.child('bus', 'c') == '[{"kind":"place","shape":"c","order":2,"to":[0,2]},{"kind":"extents","rows":1,"columns":3}]'
assert b.child('bus', 'd') == '[{"kind":"place","shape":"d","order":3,"to":[1,0]},{"kind":"extents","rows":2,"columns":3}]'
assert b.child('bus', 'e') == '[{"kind":"place","shape":"e","order":4,"to":[1,1]}]'
assert b.spec('bus', 'a', 'e', 0.0) == '[{"kind":"rule","from_shape":"a","to_shape":"e","degrees":0.0,"status":"ignored"}]'
assert b.spec('bus', 'c', 'e', 90.0) == '[{"kind":"rule","from_shape":"c","to_shape":"e","degrees":90.0,"status":"ignored"}]'
assert b.spec('bus', 'c', 'e', 180.0) == '[{"kind":"rule","from_shape":"c","to_shape":"e","degrees":180.0,"status":"ignored"}]'
assert b.queue("bus") == '[["a","e",0.0],["c","e",90.0],["c","e",180.0]]'
assert b.cells("bus") == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["b",1]],null,1,1],[[0,2],[["c",2]],null,1,1],[[1,0],[["d",3]],null,1,1],[[1,1],[["e",4]],null,1,1]]'

<a id="ADJ-06"></a>
## ADJ-06 · A grid that grows columns: east is a column on, north and south within a column; a spec that holds already is held

In [ ]:
assert b.child('', 'col') == '[{"kind":"place","shape":"col","order":4,"to":[1,1]}]'
assert b.layout('col', 2, 'columns') == '[{"kind":"grid","container":"col","span":2,"grows":"columns","cell_rotation":0.0}]'
assert b.child('col', 'a') == '[{"kind":"place","shape":"a","order":0,"to":[0,0]},{"kind":"extents","rows":1,"columns":1}]'
assert b.child('col', 'b') == '[{"kind":"place","shape":"b","order":1,"to":[1,0]},{"kind":"extents","rows":2,"columns":1}]'
assert b.child('col', 'c') == '[{"kind":"place","shape":"c","order":2,"to":[0,1]},{"kind":"extents","rows":2,"columns":2}]'
assert b.child('col', 'd') == '[{"kind":"place","shape":"d","order":3,"to":[1,1]}]'
assert b.spec('col', 'a', 'd', 90.0) == '[{"kind":"move","shape":"d","from":[1,1],"to":[0,1]},{"kind":"move","shape":"c","from":[0,1],"to":[1,1]},{"kind":"rule","from_shape":"a","to_shape":"d","degrees":90.0,"status":"met"}]'
assert b.spec('col', 'd', 'b', 0.0) == '[{"kind":"rule","from_shape":"d","to_shape":"b","degrees":0.0,"status":"ignored"}]'
assert b.spec('col', 'a', 'b', 180.0) == '[{"kind":"rule","from_shape":"a","to_shape":"b","degrees":180.0,"status":"held"}]'
assert b.cells("col") == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["d",3]],null,1,1],[[1,0],[["b",1]],null,1,1],[[1,1],[["c",2]],null,1,1]]'

<a id="ADJ-07"></a>
## ADJ-07 · Original insertion positions outlive removals and additions; a rule whose shapes move together still holds

In [ ]:
assert b.child('', 'alu') == '[{"kind":"place","shape":"alu","order":5,"to":[1,2]}]'
assert b.layout('alu', 3) == '[{"kind":"grid","container":"alu","span":3,"grows":"rows","cell_rotation":0.0}]'
assert b.child('alu', 'x') == '[{"kind":"place","shape":"x","order":0,"to":[0,0]},{"kind":"extents","rows":1,"columns":1}]'
assert b.child('alu', 'y') == '[{"kind":"place","shape":"y","order":1,"to":[0,1]},{"kind":"extents","rows":1,"columns":2}]'
assert b.child('alu', 'z') == '[{"kind":"place","shape":"z","order":2,"to":[0,2]},{"kind":"extents","rows":1,"columns":3}]'
assert b.spec('alu', 'y', 'z', 270.0) == '[{"kind":"move","shape":"z","from":[0,2],"to":[0,1]},{"kind":"move","shape":"y","from":[0,1],"to":[0,2]},{"kind":"rule","from_shape":"y","to_shape":"z","degrees":270.0,"status":"met"}]'
assert b.unchild('alu', 'x') == '[{"kind":"leave","shape":"x","from":[0,0]},{"kind":"move","shape":"z","from":[0,1],"to":[0,0]},{"kind":"move","shape":"y","from":[0,2],"to":[0,1]},{"kind":"extents","rows":1,"columns":2},{"kind":"place","shape":"x","order":6,"to":[2,0]},{"kind":"extents","rows":3,"columns":3}]'
assert b.child('alu', 'w') == '[{"kind":"place","shape":"w","order":3,"to":[0,2]},{"kind":"extents","rows":1,"columns":3}]'
assert b.unspec('alu', 'y', 'z') == '[{"kind":"rule","from_shape":"y","to_shape":"z","degrees":270.0,"status":"dropped"},{"kind":"move","shape":"y","from":[0,1],"to":[0,0]},{"kind":"move","shape":"z","from":[0,0],"to":[0,1]}]'
assert b.cells("alu") == '[[[0,0],[["y",1]],null,1,1],[[0,1],[["z",2]],null,1,1],[[0,2],[["w",3]],null,1,1]]'
assert b.child('', 'mux') == '[{"kind":"place","shape":"mux","order":7,"to":[2,1]}]'
assert b.layout('mux', 4) == '[{"kind":"grid","container":"mux","span":4,"grows":"rows","cell_rotation":0.0}]'
assert b.child('mux', 'p') == '[{"kind":"place","shape":"p","order":0,"to":[0,0]},{"kind":"extents","rows":1,"columns":1}]'
assert b.child('mux', 'q') == '[{"kind":"place","shape":"q","order":1,"to":[0,1]},{"kind":"extents","rows":1,"columns":2}]'
assert b.child('mux', 'r') == '[{"kind":"place","shape":"r","order":2,"to":[0,2]},{"kind":"extents","rows":1,"columns":3}]'
assert b.child('mux', 's') == '[{"kind":"place","shape":"s","order":3,"to":[0,3]},{"kind":"extents","rows":1,"columns":4}]'
assert b.spec('mux', 'r', 's', 90.0) == '[{"kind":"rule","from_shape":"r","to_shape":"s","degrees":90.0,"status":"held"}]'
assert b.unchild('mux', 'p') == '[{"kind":"leave","shape":"p","from":[0,0]},{"kind":"move","shape":"q","from":[0,1],"to":[0,0]},{"kind":"move","shape":"r","from":[0,2],"to":[0,1]},{"kind":"move","shape":"s","from":[0,3],"to":[0,2]},{"kind":"extents","rows":1,"columns":3},{"kind":"place","shape":"p","order":8,"to":[2,2]}]'
assert b.cells("mux") == '[[[0,0],[["q",1]],null,1,1],[[0,1],[["r",2]],null,1,1],[[0,2],[["s",3]],null,1,1]]'
assert b.rules("mux") == '[["r","s",90.0,false]]'
assert text(b.problems("cpu", "io", "bus", "col", "alu", "mux")) == '[[],[],[],[],[],[]]'

<a id="ADJ-08"></a>
## ADJ-08 · A grid's cells are turned by its cell rotation, and with them the compasses of the shapes they hold

In [ ]:
assert b.child('', 'rot') == '[{"kind":"place","shape":"rot","order":9,"to":[3,0]},{"kind":"extents","rows":4,"columns":3}]'
assert b.layout('rot', 2, 'rows', 90.0) == '[{"kind":"grid","container":"rot","span":2,"grows":"rows","cell_rotation":90.0}]'
assert b.child('rot', 'a') == '[{"kind":"place","shape":"a","order":0,"to":[0,0]},{"kind":"extents","rows":1,"columns":1}]'
assert b.child('rot', 'b') == '[{"kind":"place","shape":"b","order":1,"to":[0,1]},{"kind":"extents","rows":1,"columns":2}]'
assert b.layout('rot.a', 1, 'columns', 45.0) == '[{"kind":"grid","container":"a","span":1,"grows":"columns","cell_rotation":45.0}]'
assert b.child('rot.a', 'inner') == '[{"kind":"place","shape":"inner","order":0,"to":[0,0]},{"kind":"extents","rows":1,"columns":1}]'
L = b.store.Logical
with b.store.transaction():
    for path in ("rot.a", "rot.b", "rot.a.inner"):
        shape = b.shape(path)
        o = L.OfPoint(lambda p: p.owner(lambda a: a.insert(lambda e: e.name("origin").owner(shape)))
                      .origin_of(lambda a: a.insert(lambda e: e.shape(shape))))
        n = L.OfPoint(lambda p: p.owner(lambda a: a.insert(lambda e: e.name("n").owner(shape)))
                      .border_of(lambda a: a.insert(lambda e: e.shape(shape))))
        L.OfVector(lambda v: v.owner(lambda a: a.insert(lambda e: e.name("to n").owner(shape)))
                   .start(lambda a: a.insert(lambda e: e.point(o))).end(lambda a: a.insert(lambda e: e.point(n)))
                   .rotation(lambda r: r.degrees(90.0)).length(1.0))
a, inner = b.shape("rot.a"), b.shape("rot.a.inner")
assert text([Logical.compass(a), Logical.compass(a.owns({"name": "n"})), Logical.compass(inner)]) == '[90.0,180.0,135.0]'
assert b.spec('rot', 'a', 'b', 180.0) == '[{"kind":"rule","from_shape":"a","to_shape":"b","degrees":180.0,"status":"ignored"}]'
assert b.turn('rot', 270.0) == '[{"kind":"rotate","degrees":270.0}]'
assert text([Logical.compass(a), Logical.compass(a.owns({"name": "n"})), Logical.compass(inner)]) == '[270.0,0.0,315.0]'

<a id="ADJ-09"></a>
## ADJ-09 · The journal, replayed in a fresh store, makes the same adjustments

In [ ]:
assert text(b.replay()) == '[]'